In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statistics
import csv
import copy
import pickle

from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
import openml

from tqdm import tqdm

from itertools import product, cycle
from functools import partial

from sklearn.base import clone as sk_clone

from helpers.persistence import save_var, load_var
from helpers.progress_bar import ProgressBar
from helpers.openml_data import tabular_id_list
from helpers.openml_data_v2 import our_new_list
from helpers.baseline import train_and_test as train_test_clf

from joblib import Parallel, delayed

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
settings = {
    'method': 'bootstrap',
    'kfolds': 5,
}

In [3]:
pbar = False

In [4]:
def set_random_seeds():
    np.random.seed(0)
    
set_random_seeds()

In [5]:
def find_best_params(params_range, clf, train_df, train_y, valid_df, valid_y):
    best_params = False
    best_val = 0
        
    param_combinations = list(product(*params_range.values()))
        
    for i, params in enumerate(param_combinations):
        params = {k:v for k,v in zip(params_range.keys(), params)}
        
            # clone to get the unfitted yet a true copy of the classifier
            # didn't change the output
        set_random_seeds()
        model =  sk_clone(clf)
        model.set_params(**params)
        
            
        model.fit(train_df, train_y)
            
            
        y_pred = model.predict(valid_df)
        score = f1_score(valid_y, y_pred, average='weighted')
            
        pbar and pbar.set_description(f'Validating: {i+1} / {len(param_combinations)} | val: {score:.3f} (best={best_val:.3f})')
            
        if score > best_val:
            best_val = score
            best_params = params
    return best_params

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def train_and_test(dataset_name, pbar=False):
    dataset = openml.datasets.get_dataset(dataset_name)

    X_raw, y, categorical_indicator, attribute_names = dataset.get_data(
        dataset_format="array", target=dataset.default_target_attribute
    )
    df = pd.DataFrame(X_raw, columns=attribute_names)
    cat_mask = np.array(categorical_indicator)

    numeric_features = df.columns[~cat_mask]
    categorical_features = df.columns[cat_mask]

    numeric_transformer = Pipeline(
        steps=[
            # ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            # ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder='passthrough'
    )
    
    X_ohe = preprocessor.fit_transform(df)
    df_ohe = pd.DataFrame(X_ohe, columns=preprocessor.get_feature_names_out(), index=df.index)
    
    # display(df_ohe)
    numeric_features = [f'remainder__{x}' for x in numeric_features]
    
    num_scaler = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
        ],
        remainder='passthrough'
    )

    params_range = {
        "classifier__n_estimators": [50, 80, 110],
        "classifier__max_depth": [2, 5, 10, 15]
    }

    clf = Pipeline(
        steps=[
            ("num_scaler", num_scaler),
            # ("scaler", StandardScaler()),
            ("classifier", GradientBoostingClassifier(
                n_estimators=100, random_state=42)),
        ]
    )

    k_folds = settings['kfolds']
    cv = StratifiedKFold(n_splits=k_folds, random_state=42, shuffle=True)

    pbar and pbar.add_prefix('creating folds')
    fold_limit = 5 if settings['method'] == 'kfold' else 30

    kfold_splits = list(cv.split(df, y))

    scores = []
    y_preds = []
    y_trues = []
    best_params_list = []
    fold_times = []

    # run k fold in loop
    for fold_counter in range(fold_limit):
        start_time = time()
        pbar and pbar.edit_last_prefix(
            f'fold={fold_counter+1}/{fold_limit} | ')

        if settings['method'] == 'kfold':
            train, test = kfold_splits[fold_counter]

            # divide the data for train and test
            train_df, train_y = df.iloc[train], y[train]
            test_df, test_y = df.iloc[test], y[train]
        else:
            train_df, test_df, train_y, test_y = train_test_split(df_ohe, y,
                                                                  stratify=y,
                                                                  shuffle=True,
                                                                  test_size=1/5,
                                                                  random_state=fold_counter)

        train_df, valid_df, train_y, valid_y = train_test_split(train_df, train_y,
                                                                stratify=train_y,
                                                                test_size=1/8, random_state=42)
        
        pbar and pbar.set_description('validating')
#         best_params = False
#         best_val = 0
        
#         param_combinations = list(product(*params_range.values()))
        
#         for i, params in enumerate(param_combinations):
            
#             params = {k:v for k,v in zip(params_range.keys(), params)}
        
#             # clone to get the unfitted yet a true copy of the classifier
#             # didn't change the output
#             set_random_seeds()
#             model =  sk_clone(clf)
#             model.set_params(**params)
        
            
#             model.fit(train_df, train_y)
            
            
#             y_pred = model.predict(valid_df)
#             score = f1_score(valid_y, y_pred, average='weighted')
            
#             pbar and pbar.set_description(f'Validating: {i+1} / {len(param_combinations)} | val: {score:.3f} (best={best_val:.3f})')
            
#             if score > best_val:
#                 best_val = score
#                 best_params = params
        
        
#         # clone to get the unfitted yet a true copy of the classifier
#         # didnt change the output
#         set_random_seeds()
#         model =  sk_clone(clf)
#         model.set_params(**best_params)
        
#         model.fit(train_df, train_y)
        
        
#         y_pred = model.predict(test_df)
        
#         score = f1_score(test_y, y_pred, average='weighted')

        score, best_params, y_pred = train_test_clf(clf, params_range, 
                                                    (train_df, train_y),
                                                    (valid_df, valid_y),
                                                    (test_df, test_y), 
                                                    n_jobs = 4,
                                                   )
    
        time_taken = time() - start_time
        
        scores.append(score)
        y_preds.append(y_pred)
        y_trues.append(test_y)
        best_params_list.append(best_params)
        fold_times.append(time_taken)


    results = {
        'scores': scores,
        'predicted': y_preds,
        'actual': y_trues,
        'best_params': best_params_list,
        'fold_times': fold_times,
        'total_time': np.sum(fold_times),
    }

    save_var(results, f'./saved_vars/gbt_{dataset_name}.pkl')

    return scores

In [7]:
bool({}) == True, bool({'a': 2}) == True

(False, True)

In [8]:
def get_results(id, pbar=False):
    rows = []
    results = load_var(f'./saved_vars/gbt_{id}.pkl', False) or {}
    # results = {} # do new
    
    scores = train_and_test(id, pbar=pbar) if not bool(results) else results['scores']

    for i, s in enumerate(scores):
        rows.append([id, 'gbt', i, s])

    return pd.DataFrame(rows, columns=['id','corruption','fold','test_score'])

In [9]:
from time import time
start_time, end_time, time_taken = 0, 0, 0

In [10]:

id_list = our_new_list #+ [151, 3]
# id_list = np.setdiff1d(tabular_id_list, id_list)

pbar = ProgressBar(id_list)
# df_list = Parallel(n_jobs=10)(delayed(get_results)(id)
#                                      for id in id_list) 

df_list = []
for id in pbar:
    pbar.clear_prefix()
    if id in [1478, 41166]:
        continue
    pbar.add_prefix(f'Doing id={id}')
    start_time = time()
    r = get_results(id, pbar=pbar)
    end_time = time()
    time_taken = end_time - start_time
    # print(time_taken, time_taken/30)
    df_list.append(r)


df = pd.concat(df_list)
df.to_csv('./exports/gbt.csv')

100%|████████████████████████████████████████████████████████████████████| 28/28 [00:00<00:00, 732.60it/s]


In [11]:
df.groupby('id').test_score.agg(['mean','std'])

,mean,std
id,,
11,0.848506,0.019099
23,0.541930,0.029927
31,0.724965,0.032756
37,0.749383,0.031328
46,0.960616,0.007591
50,0.983612,0.011746
54,0.735734,0.032636
458,0.978327,0.012145
469,0.194197,0.023007


In [12]:
large_dataset = [151, 1461, 40668, 41027]

In [13]:
tmp = df.groupby('id').test_score.agg(['mean','std']).reset_index()
df0 = pd.read_csv('./exports/tabular_description.csv')

df0.set_index('id').join(tmp.set_index('id')).dropna()

,name,samples,features,categories,classes,mean,std
id,,,,,,,
11,balance-scale,625,4,0,3,0.848506,0.019099
23,cmc,1473,9,7,3,0.541930,0.029927
31,credit-g,1000,20,13,2,0.724965,0.032756
37,diabetes,768,8,0,2,0.749383,0.031328
46,splice,3190,61,60,3,0.960616,0.007591
50,tic-tac-toe,958,9,9,2,0.983612,0.011746
54,vehicle,846,18,0,4,0.735734,0.032636
458,analcatdata_authorship,841,70,0,4,0.978327,0.012145
469,analcatdata_dmft,797,4,4,6,0.194197,0.023007
